# Saber 11: ¿puede el contexto socioeconómico predecir el desempeño académico?

**Tarea Cloud Computing 4 Data Science · Magíster UAI 2026**

**La pregunta de fondo:** ¿cuánto pesa el contexto socioeconómico de un estudiante en su resultado académico?

**La pregunta del modelo (clasificación):** ¿quedará este estudiante sobre o bajo la mediana nacional en matemáticas, conociendo solo su contexto — estrato, educación de los padres, internet en casa, tipo de colegio?

**Fuente:** ICFES (Instituto Colombiano para la Evaluación de la Educación) · "Resultados únicos Saber 11" · datos.gov.co (fuente estatal, oficial y primaria) · https://www.datos.gov.co/d/kgxf-xxbe

**Privacidad — minimización de datos desde el origen:** el dataset original trae cuasi-identificadores (ID del estudiante, fecha de nacimiento, colegio exacto, municipio). Nuestra extracción usa `$select` en la API para traer **solo** las columnas que el modelo necesita: ninguna identifica a una persona. El pipeline nunca toca esos campos.

Muestra de trabajo: **150.000 estudiantes del período 2022-4** (de 1.065.888 del período; 7,1M en total).

*Requisitos: `pip install pandas pyarrow matplotlib`*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

# El parquet debe estar en la misma carpeta que este notebook
df = pd.read_parquet("saber11_muestra.parquet")
print(f"Filas: {len(df):,} · Columnas: {len(df.columns)}")
df.head(3)

## 1 · Vista general

Cada fila es un estudiante: su puntaje de matemáticas e inglés, y su contexto. Ninguna columna lo identifica.

*(Nota para la defensa: en la API JSON los puntajes llegan como texto — "88", no 88. Aquí ya vienen tipificados porque la extracción usó el export CSV, pero la regla del curso aplica igual: tipificar es tarea del pipeline, no de la fuente.)*

In [ ]:
df.dtypes

In [ ]:
# ¿Cómo se distribuye el puntaje de matemáticas?
print(df["punt_matematicas"].describe().round(1))

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(df["punt_matematicas"], bins=50, color="#0E6E6B")
ax.axvline(df["punt_matematicas"].median(), color="#B5772A", linewidth=2)
ax.set_title("Distribución del puntaje de matemáticas · la línea es la mediana")
plt.tight_layout(); plt.show()

## 2 · Limpieza: los nulos son parte del fenómeno

Los datos socioeconómicos son autorreportados: hay estudiantes sin estrato, sin educación de los padres declarada. **No los botamos** — los marcamos como "Sin dato", porque no responder también dice algo del contexto (ya vimos que los "Sin Estrato" tienen el peor promedio).

In [ ]:
print("Nulos por columna:")
print(df.isna().sum())

cat_cols = df.select_dtypes(include="object").columns
df[cat_cols] = df[cat_cols].fillna("Sin dato")
print("\nNulos restantes en categóricas:", df[cat_cols].isna().sum().sum())

## 3 · Exploración: las brechas que el modelo va a aprender

### 3.1 · Internet en casa

In [ ]:
display(df.groupby("fami_tieneinternet")["punt_matematicas"].agg(["mean", "count"]).round(1))
# Lectura: ~6 puntos de brecha por una condición del hogar.

### 3.2 · Tipo de colegio y área

In [ ]:
display(df.groupby("cole_naturaleza")["punt_matematicas"].agg(["mean", "count"]).round(1))
display(df.groupby("cole_area_ubicacion")["punt_matematicas"].agg(["mean", "count"]).round(1))

### 3.3 · Estrato y educación de la madre

In [ ]:
orden_estrato = ["Sin dato", "Sin Estrato", "Estrato 1", "Estrato 2", "Estrato 3", "Estrato 4", "Estrato 5", "Estrato 6"]
por_estrato = df.groupby("fami_estratovivienda")["punt_matematicas"].mean().reindex(orden_estrato)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(range(len(por_estrato)), por_estrato.values, color="#0E6E6B")
ax.set_xticks(range(len(por_estrato)))
ax.set_xticklabels(por_estrato.index, rotation=20, ha="right", fontsize=9)
ax.set_title("Promedio de matemáticas por estrato de la vivienda")
plt.tight_layout(); plt.show()

print("\nPor educación de la madre (top y bottom):")
em = df.groupby("fami_educacionmadre")["punt_matematicas"].agg(["mean", "count"]).round(1).sort_values("mean")
display(pd.concat([em.head(3), em.tail(3)]))

## 4 · Construcción del target: `rendimiento_alto`

**La decisión de diseño** (hay que poder defenderla): un estudiante es de *rendimiento alto* si su puntaje de matemáticas queda **en o sobre la mediana nacional** de la muestra. Con la mediana, las clases quedan balanceadas por construcción (~50/50) — el escenario más limpio para entrenar y comparar modelos.

La clave del enunciado: el modelo **no sabe nada del estudiante como estudiante** — ni notas previas ni asistencia. Solo su contexto. Si aun así predice bien, ese es el hallazgo.

In [ ]:
mediana = df["punt_matematicas"].median()
df["rendimiento_alto"] = (df["punt_matematicas"] >= mediana).astype(int)

print(f"Mediana nacional (muestra): {mediana}")
print("Balance de clases:")
print(df["rendimiento_alto"].value_counts(normalize=True).round(3))

In [ ]:
# Sanity check: la tasa de 'rendimiento alto' debe crecer con el estrato
tasa = df.groupby("fami_estratovivienda")["rendimiento_alto"].mean().reindex(orden_estrato) * 100

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(len(tasa)), tasa.values, color="#B5772A")
ax.set_xticks(range(len(tasa)))
ax.set_xticklabels(tasa.index, rotation=20, ha="right", fontsize=9)
ax.axhline(50, color="#555", linewidth=1, linestyle="--")
ax.set_title("% de estudiantes sobre la mediana, por estrato · la línea es el 50%")
plt.tight_layout(); plt.show()

# Si la escalera sube de izquierda a derecha, el target captura la brecha que queremos estudiar.

## 5 · Guardar el dataset para el modelo

Solo features de contexto + target. El puntaje NO va como feature (sería filtrar la respuesta).

In [ ]:
FEATURES = [
    "fami_estratovivienda",   # estrato de la vivienda
    "fami_educacionmadre",    # educación de la madre
    "fami_educacionpadre",    # educación del padre
    "fami_tieneinternet",     # internet en casa
    "fami_tienecomputador",   # computador en casa
    "cole_naturaleza",        # colegio oficial / no oficial
    "cole_area_ubicacion",    # urbano / rural
    "cole_jornada",           # completa / mañana / tarde / noche
    "estu_genero",            # género
    "estu_depto_reside",      # departamento (no municipio: minimización)
]
TARGET = "rendimiento_alto"

dataset = df[FEATURES + [TARGET]].copy()
dataset.to_parquet("dataset_modelo_saber11.parquet", index=False)
print(f"Guardado dataset_modelo_saber11.parquet: {len(dataset):,} filas × {len(dataset.columns)} columnas")
dataset.head()

## Próximos pasos (frente Modelo)

1. `train_test_split` estratificado (80/20, `random_state=42`).
2. One-hot encoding de las categóricas (`pd.get_dummies` o `OneHotEncoder`).
3. **Regresión logística** como línea base (patrón del Taller 1).
4. **Random Forest** para comparar + importancia de variables (responde: ¿qué pesa más — estrato, educación de la madre, internet?).
5. Métricas: matriz de confusión, precision/recall, ROC AUC.
6. Exportar: `joblib.dump(modelo, "modelo_saber11.pkl")`.
7. App FastAPI con `/predict`: entra el perfil de contexto, sale la probabilidad de quedar sobre la mediana (patrón `Model_04_FASTAPI.py` del Taller 1).